# Tách câu song ngữ Thái – Việt từ PDF thành dataset

Notebook đọc file PDF có **lớp chữ**, trong đó có một khối tiếng Thái (viết bằng chữ Latinh) và một khối bản dịch tiếng Việt, rồi:

1. Đọc từng dòng kèm vị trí trên trang, ghép thành **đoạn văn** (dựa vào thụt đầu dòng và khoảng cách dòng).
2. Nhận diện đoạn nào là tiếng Việt, đoạn nào là tiếng Thái (bằng từ phổ biến của tiếng Việt, và dùng khoảng trống lớn trên trang để tìm ranh giới với các đoạn ngắn như tiêu đề).
3. **Gióng đoạn** Thái với đoạn Việt, rồi **gióng câu** trong từng cặp đoạn, bằng phương pháp độ dài ký tự (Gale–Church). Cách này không cần mô hình ngôn ngữ nên dùng được cho tiếng Thái Đen, vốn không có trong LaBSE hay LASER.
4. Xuất bảng: cột `thai`, cột `vietnamese`, cột `type` (1-1, 1-2, 2-1...) cho biết câu nào ghép 1 với 1 và câu nào phải gộp.

**Cách dùng:** gắn dataset chứa PDF vào notebook (Add Input), rồi bấm **Run All**. Cần bật **Settings → Internet → On** để cài thư viện. Kết quả nằm ở tab **Output**: `thaiden_viet_dataset.xlsx` (3 sheet) và `thaiden_viet_dataset.csv`.

**Lưu ý:** các trang viết bằng font chữ Thái cũ (ví dụ `ThaiVietNam...`) không có bảng Unicode nên chữ trích ra bị loạn. Notebook tự bỏ qua phần đó và báo ở sheet `bo_qua`.

In [ ]:
# Cài thư viện (cần Internet: Settings -> Internet -> On)
!pip install -q pdfplumber python-docx

In [ ]:
import math
import re
import unicodedata
from pathlib import Path
from statistics import median

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# ===== Cấu hình =====
INPUT_FILE = None  # None = tự tìm trong /kaggle/input. Hoặc ghi đường dẫn, ví dụ "/kaggle/input/ten-dataset/file.pdf"
KAGGLE_INPUT = Path("/kaggle/input")
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("output")
OUT_DIR.mkdir(exist_ok=True)
EXTS = {".pdf", ".docx", ".txt"}

# Font chữ Thái đời cũ (không có bảng Unicode) -> chữ trích ra bị loạn nên bỏ qua. Thêm tên font khác vào đây nếu gặp.
SKIP_FONT_KEYWORDS = ("ThaiVietNam",)
SHOW_DEBUG = True   # in danh sách đoạn và nhãn ngôn ngữ để bạn kiểm tra

# Từ phổ biến của tiếng Việt, dùng để phân biệt bản dịch tiếng Việt với chữ Thái Latinh hoá
VI_WORDS = set("""của và những không một người được các này tôi có đã cho để trong với là nhiều rất thì mới cũng
nhưng rồi lại ở về vào khi qua sau thêm theo như từ bởi nên nếu mà sẽ đang hay hoặc vì cái nào ông bà anh em chúng
họ nó đó đây rằng cần phải biết làm xin chân thành cảm tất cả mọi hơn chưa đủ""".split())
THAI_SCRIPT = re.compile(r"[\u0E00-\u0E7F]")
ABBR = re.compile(r"(?i)\b(?:ts|ths|pgs|gs|bs|ks|tp|tr|vd|dr|mr|mrs|ms|prof|st|vs|fig|no|cf)\.$")


# ===== Đọc file -> danh sách đoạn: {"text", "gap"} (gap = khoảng trống phía trên, tính theo chiều cao dòng) =====
def clean(text):
    return unicodedata.normalize("NFC", re.sub(r"\s+", " ", text)).strip()


def read_pdf(path):
    import pdfplumber
    paras, skipped, cur = [], [], None
    with pdfplumber.open(str(path)) as pdf:
        for pno, page in enumerate(pdf.pages, start=1):
            chars, n_skip = [], 0
            for c in page.chars:
                if any(k.lower() in c.get("fontname", "").lower() for k in SKIP_FONT_KEYWORDS):
                    n_skip += bool(c["text"].strip())   # bỏ hẳn ký tự của font cũ (kể cả dấu cách, vì toạ độ của chúng hay lệch dòng)
                    continue
                chars.append(c)
            if n_skip:
                skipped.append((path.name, pno, n_skip))
            # gom ký tự thành dòng theo toạ độ dọc
            chars.sort(key=lambda c: (round(c["top"]), c["x0"]))
            rows = []
            for c in chars:
                if rows and abs(c["top"] - rows[-1][0]["top"]) < 3:
                    rows[-1].append(c)
                else:
                    rows.append([c])
            explicit_spaces = sum(c["text"] == " " for c in chars) > 0.05 * max(len(chars), 1)
            lines = []
            for row in rows:
                row.sort(key=lambda c: c["x0"])
                if explicit_spaces:      # PDF có ký tự dấu cách thật -> tin vào nó (tránh tách đôi từ do sai lệch toạ độ)
                    parts = []           # chỉ bù thêm dấu cách ở khoảng hở cỡ 1 dấu cách (0.2-1.0 cỡ chữ, kể cả dấu cách bị giãn khi canh đều hai bên) mà không có ký tự dấu cách
                    for k, c in enumerate(row):
                        if k and c["text"] != " " and row[k - 1]["text"] != " ":
                            if 0.2 * c.get("size", 10) <= c["x0"] - row[k - 1]["x1"] <= 1.0 * c.get("size", 10):
                                parts.append(" ")
                        parts.append(c["text"])
                    text = "".join(parts)
                else:                    # PDF không có dấu cách -> đoán theo khoảng hở
                    text = "".join((" " if k and c["x0"] - row[k - 1]["x1"] > 0.2 * c.get("size", 10) else "") + c["text"]
                                   for k, c in enumerate(row))
                text = clean(text)
                if not any(ch.isalpha() for ch in text):    # số trang, ký hiệu
                    continue
                body = [c for c in row if c["text"].strip()]
                lines.append({"text": text, "x0": min(c["x0"] for c in body),
                              "top": min(c["top"] for c in body), "bottom": max(c["bottom"] for c in body)})
            if not lines:
                continue
            long_lines = [l["x0"] for l in lines if len(l["text"]) >= 40]
            margin = min(long_lines) if long_lines else min(l["x0"] for l in lines)
            line_h = median(l["bottom"] - l["top"] for l in lines) or 12
            prev_bottom = None
            for l in lines:
                indent = l["x0"] - margin
                gap = (l["top"] - prev_bottom) / line_h if prev_bottom is not None else 1.5
                if cur is None or indent > 8 or gap > 0.8:       # thụt đầu dòng / canh giữa / cách dòng lớn = đoạn mới
                    if cur:
                        paras.append(cur)
                    cur = {"text": l["text"], "gap": gap}
                else:
                    cur["text"] += " " + l["text"]
                prev_bottom = l["bottom"]
    if cur:
        paras.append(cur)
    return paras, skipped


def read_paragraphs(path):
    ext = path.suffix.lower()
    if ext == ".pdf":
        return read_pdf(path)
    if ext == ".docx":
        from docx import Document
        lines = [p.text for p in Document(str(path)).paragraphs]
    else:
        lines = path.read_text(encoding="utf-8-sig").splitlines()
    paras, blank = [], True          # file không có toạ độ: dòng trống phía trên được coi là khoảng trống lớn
    for l in lines:
        t = clean(l)
        if not t:
            blank = True
            continue
        paras.append({"text": t, "gap": 2.0 if blank else 1.0})
        blank = False
    return paras, []


# ===== Nhận diện ngôn ngữ từng đoạn: "src" (Thái) hoặc "vi" (Việt) =====
def vi_score(text):
    toks = re.findall(r"[^\W\d_]+", text.lower())
    return (sum(t in VI_WORDS for t in toks) / len(toks) if toks else 0.0), len(toks)


def label_paragraphs(paras):
    labels = []
    for p in paras:
        t = p["text"]
        score, n = vi_score(t)
        if len(THAI_SCRIPT.findall(t)) > 0.3 * len(t):
            labels.append("src")                    # chữ Thái Unicode
        elif n >= 6 and score >= 0.15:
            labels.append("vi")
        elif n >= 6 and score <= 0.06:
            labels.append("src")
        else:
            labels.append(None)                     # đoạn ngắn / chưa chắc -> quyết định theo ngữ cảnh
    i, n = 0, len(paras)
    while i < n:
        if labels[i] is not None:
            i += 1
            continue
        j = i
        while j < n and labels[j] is None:
            j += 1
        left = labels[i - 1] if i > 0 else None
        right = labels[j] if j < n else None
        if left and right and left != right:
            # ranh giới Thái/Việt nằm ở chỗ có khoảng trống dọc lớn nhất trong nhóm đoạn chưa chắc
            k = max(range(i, j + 1), key=lambda t: paras[t]["gap"])
            for t in range(i, j):
                labels[t] = left if t < k else right
        else:
            for t in range(i, j):
                labels[t] = left or right or "src"
        i = j
    return labels


def to_runs(paras, labels):
    runs = []
    for p, lg in zip(paras, labels):
        if runs and runs[-1][0] == lg:
            runs[-1][1].append(p["text"])
        else:
            runs.append([lg, [p["text"]]])
    return runs


# ===== Gióng theo độ dài ký tự (Gale & Church, 1993) =====
S2 = 6.8
PRIOR = {(1, 1): 0.89, (1, 0): 0.0099, (0, 1): 0.0099, (2, 1): 0.089, (1, 2): 0.089, (2, 2): 0.011}


def _match_cost(l1, l2, c):
    if l1 == 0 and l2 == 0:
        return 0.0
    mean = (l1 + l2 / c) / 2
    z = (l2 - l1 * c) / math.sqrt(max(mean, 1.0) * S2)
    p = max(2 * (1 - 0.5 * (1 + math.erf(abs(z) / math.sqrt(2)))), 1e-12)
    return -math.log(p)


def align_lengths(a, b, c=1.0):
    """a, b: độ dài (ký tự) của hai dãy. Trả về [(chỉ số trong a, chỉ số trong b), ...] theo thứ tự."""
    n, m = len(a), len(b)
    INF = float("inf")
    D = [[INF] * (m + 1) for _ in range(n + 1)]
    back = [[None] * (m + 1) for _ in range(n + 1)]
    D[0][0] = 0.0
    for i in range(n + 1):
        for j in range(m + 1):
            if D[i][j] == INF:
                continue
            for (di, dj), pr in PRIOR.items():
                ni, nj = i + di, j + dj
                if ni > n or nj > m:
                    continue
                cost = D[i][j] - math.log(pr) + _match_cost(sum(a[i:ni]), sum(b[j:nj]), c)
                if cost < D[ni][nj]:
                    D[ni][nj], back[ni][nj] = cost, (i, j)
    beads, i, j = [], n, m
    while (i, j) != (0, 0):
        pi, pj = back[i][j]
        beads.append((list(range(pi, i)), list(range(pj, j))))
        i, j = pi, pj
    return beads[::-1]


# ===== Tách câu (theo dấu câu, có xử lý từ viết tắt) =====
def split_sentences(text):
    pieces = [p.strip() for p in re.split(r'(?:(?<=[.!?…])|(?<=[.!?…]["”’)]))\s+', text) if p.strip()]
    out = []
    for p in pieces:
        if out and ABBR.search(out[-1]):
            out[-1] += " " + p
        else:
            out.append(p)
    return out


def process(path):
    paras, skipped = read_paragraphs(path)
    labels = label_paragraphs(paras)
    if SHOW_DEBUG:
        print(f"--- {path.name}: {len(paras)} đoạn")
        for p, lg in zip(paras, labels):
            print(f"  [{lg:>3}] {len(p['text']):4d} ký tự | {p['text'][:60]}")
    runs = to_runs(paras, labels)
    tot = {"src": 0, "vi": 0}
    for lg, texts in runs:
        tot[lg] += sum(len(t) for t in texts)
    c = min(2.0, max(0.5, tot["vi"] / tot["src"])) if tot["src"] and tot["vi"] else 1.0
    ok, review, pair_no = [], [], 0
    for r in range(0, len(runs) - 1, 2):           # các khối xen kẽ: Thái, Việt, Thái, Việt, ...
        (lg1, t1), (lg2, t2) = runs[r], runs[r + 1]
        src_p, vi_p = (t1, t2) if lg1 == "src" else (t2, t1)
        for ia, ib in align_lengths([len(t) for t in src_p], [len(t) for t in vi_p], c):
            pair_no += 1
            s_text = " ".join(src_p[k] for k in ia)
            v_text = " ".join(vi_p[k] for k in ib)
            if not ia or not ib:
                review.append((path.name, pair_no, "đoạn không có đoạn đối ứng", s_text, v_text))
                continue
            s_sents, v_sents = split_sentences(s_text), split_sentences(v_text)
            for sa, sb in align_lengths([len(s) for s in s_sents], [len(s) for s in v_sents], c):
                s_join = " ".join(s_sents[k] for k in sa)
                v_join = " ".join(v_sents[k] for k in sb)
                if not sa or not sb:
                    review.append((path.name, pair_no, "câu không có câu đối ứng", s_join, v_join))
                else:
                    ok.append((s_join, v_join, f"{len(sa)}-{len(sb)}", round(len(s_join) / len(v_join), 2),
                               path.name, pair_no))
    if len(runs) % 2:
        print(f"{path.name}: khối cuối không có khối đối ứng, đã bỏ qua.")
    return ok, review, skipped

In [ ]:
if INPUT_FILE:
    files = [Path(INPUT_FILE)]
elif KAGGLE_INPUT.exists():
    files = sorted(p for p in KAGGLE_INPUT.rglob("*") if p.suffix.lower() in EXTS)
else:
    files = []
if not files:
    raise SystemExit("Không thấy file .pdf/.docx/.txt nào. Hãy gắn dataset chứa PDF vào notebook (Add Input) rồi chạy lại.")

ok_all, review_all, skipped_all = [], [], []
for f in files:
    ok, review, skipped = process(f)
    print(f"{f.name}: {len(ok)} cặp câu, {len(review)} mục cần kiểm tra, {len(skipped)} trang có chữ font cũ bị bỏ qua")
    ok_all += ok
    review_all += review
    skipped_all += skipped

df = pd.DataFrame(ok_all, columns=["thai", "vietnamese", "type", "len_ratio", "source", "paragraph"])
df.insert(0, "id", range(1, len(df) + 1))
review = pd.DataFrame(review_all, columns=["source", "paragraph", "ly_do", "thai", "vietnamese"])
skipped_df = pd.DataFrame(skipped_all, columns=["source", "page", "so_ky_tu_font_cu_bi_bo"])
print(f"\nTổng: {len(df)} cặp câu, {len(review)} mục cần kiểm tra thủ công.")
print("Phân bố loại ghép:", df["type"].value_counts().to_dict())

In [ ]:
xlsx_path = OUT_DIR / "thaiden_viet_dataset.xlsx"
with pd.ExcelWriter(xlsx_path) as writer:
    df.to_excel(writer, sheet_name="dataset", index=False)
    review.to_excel(writer, sheet_name="can_kiem_tra", index=False)
    skipped_df.to_excel(writer, sheet_name="bo_qua", index=False)
df.to_csv(OUT_DIR / "thaiden_viet_dataset.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:", xlsx_path, "và", OUT_DIR / "thaiden_viet_dataset.csv")

pd.set_option("display.max_colwidth", 80)
print("\n=== dataset ===")
display(df)
print("\n=== cần kiểm tra thủ công ===")
display(review)
print("\n=== trang có chữ font cũ bị bỏ qua ===")
display(skipped_df)